In [0]:
from pyspark.sql.functions import col, when, current_date
from delta.tables import DeltaTable

In [0]:
%sql
USE CATALOG liquid_telecom;

In [0]:
df_fact_silver = spark.read.table("liquid_telecom.silver.s_fact_subscriptions_revenue")

In [0]:
df_dim_product = spark.read.table("liquid_telecom.gold.g_dim_product")
df_dim_account = spark.read.table("liquid_telecom.gold.g_dim_account")
df_dim_date = spark.read.table("liquid_telecom.gold.g_dim_date")

In [0]:
df_gold_stage = df_fact_silver \
    .join(df_dim_product, "product_id", "left") \
    .join(df_dim_account, "account_id", "left") \
    .join(
        df_dim_date,
        df_fact_silver.order_date == df_dim_date.date,
        "left"
    )

### **Creating calculated columns**

In [0]:
df_gold_stage = df_gold_stage \
    .withColumn(
        "total_revenue_usd",
        col("total_one_off_price_usd") + col("total_recurring_price_usd")
    ) \
    .withColumn(
        "revenue_type",
        when(
            (col("total_one_off_price_usd") > 0) & (col("total_recurring_price_usd") > 0),
            "Mixed"
        ).when(
            col("total_one_off_price_usd") > 0,
            "One-Time"
        ).when(
            col("total_recurring_price_usd") > 0,
            "Recurring"
        ).otherwise("Unknown")
    ) \
    .withColumn(
        "is_recurring_flag",
        when(col("total_recurring_price_usd") > 0, 1).otherwise(0)
    ) \
    .withColumn("ingestion_date", current_date())

### **Final required gold fact table**

In [0]:
df_gold_final = df_gold_stage.select(
    "product_id",
    "account_id",
    "date_key",
    "account_created_date",
    "order_date",
    "operating_country",
    "total_one_off_price_usd",
    "total_recurring_price_usd",
    "total_contract_value_usd",
    "total_revenue_usd",
    "revenue_type",
    "is_recurring_flag",
    "ingestion_date"
)

In [0]:
display(df_gold_final.limit(10))

product_id,account_id,date_key,account_created_date,order_date,operating_country,total_one_off_price_usd,total_recurring_price_usd,total_contract_value_usd,total_revenue_usd,revenue_type,is_recurring_flag,ingestion_date
P000001,A000000000008,20240512,2024-05-12,2024-05-12,LTSAT - Liquid Telecommunications Satellite Services,0.00,100.00,1200.00,100.00,Recurring,1,2026-01-30
P000002,A000000000022,20241016,2024-10-13,2024-10-16,LTZM - Liquid Telecommunications Zambia Ltd,50.50,25.16,352.42,75.66,Mixed,1,2026-01-30
P000002,A000000000477,20241003,2024-09-10,2024-10-03,LTZM - Liquid Telecommunications Zambia Ltd,43.38,0.00,43.38,43.38,One-Time,0,2026-01-30
P000003,A000000000008,20240612,2024-06-09,2024-06-12,LTSAT - Liquid Telecommunications Satellite Services,0.00,275.00,3300.00,275.00,Recurring,1,2026-01-30
P000004,A000000000008,20240512,2024-05-12,2024-05-12,LTSAT - Liquid Telecommunications Satellite Services,0.00,275.00,3300.00,275.00,Recurring,1,2026-01-30
P000005,A000000000047,20241030,2024-10-27,2024-10-30,LTK - Liquid Telecommunications Kenya Ltd.,270.00,0.00,270.00,270.00,One-Time,0,2026-01-30
P000006,A000000000048,20250102,2024-12-29,2025-01-02,LTZ - Liquid Telecom Zimbabwe,13440.00,10500.00,139440.00,23940.00,Mixed,1,2026-01-30
P000007,A000000000008,20240417,2024-04-16,2024-04-17,LTSAT - Liquid Telecommunications Satellite Services,0.00,40.00,480.00,40.00,Recurring,1,2026-01-30
P000007,A000000000008,20240612,2024-06-09,2024-06-12,LTSAT - Liquid Telecommunications Satellite Services,0.00,80.00,960.00,80.00,Recurring,1,2026-01-30
P000008,A000000000054,20250424,2025-03-04,2025-04-24,LTZM - Liquid Telecommunications Zambia Ltd,462.50,1387.50,17112.50,1850.00,Mixed,1,2026-01-30


### **writing in delta format**

In [0]:
catalog = "liquid_telecom"
database = "gold"
table = "g_fact_subscriptions_revenue"

full_table_name = f"{catalog}.{database}.{table}"

if not spark.catalog.tableExists(full_table_name):
    print("Creating Gold fact table")

    df_gold_final.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(full_table_name)

else:
    print("Merging into existing Gold fact table")

    deltaTable = DeltaTable.forName(spark, full_table_name)

    deltaTable.alias("gold") \
        .merge(
            df_gold_final.alias("stage"),
            """
            gold.product_id = stage.product_id
            AND gold.account_id = stage.account_id
            AND gold.date_key = stage.date_key
            """
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()


Merging into existing Gold fact table
